# **PAC (Probably Approximately Correct)**

Prerequisites: ERM (`07_ML_Theory/01_hypothesis_spaces_and_erm.ipynb`),
Bias-Variance Tradeoff.

## 1. Shattering and VC Dimension - formal definition

A hypothesis class $\mathcal H$ **shatters** a set of points $\{x_1,\dots,x_m\}$
if, for every one of the $2^m$ possible labelings of those points, some
$h\in\mathcal H$ achieves it exactly. The **VC dimension** of $\mathcal H$
is the size of the *largest* set it can shatter.

## 2. Worked Proof - VC dimension of a 2D linear classifier is 3

**3 points CAN be shattered** (if not collinear): take any 3 non-collinear
points. There are $2^3=8$ labelings. Two are trivial (all same class -
any line not touching the points works). The remaining 6 put one point
against the other two, or split 1-vs-2 in various ways - for 3 points in
general position, a straight line can always be drawn separating any one
point (or any split) from the rest, since 3 non-collinear points can
always be partitioned by *some* line for *every* one of the 8 labelings -
verify by literally sketching all 8 cases on paper for a specific triangle,
e.g. $(0,0),(1,0),(0,1)$.

**4 points CANNOT always be shattered** - the counterexample: 4 points in
"XOR" configuration, $A=(0,0)$ class 1, $B=(1,1)$ class 1, $C=(0,1)$ class
0, $D=(1,0)$ class 0. No single straight line separates $\{A,B\}$ from
$\{C,D\}$ (proved computationally below - a `LinearSVC` fit on this exact
configuration achieves only 50% training accuracy, i.e. chance level,
confirming no linear separator exists for this labeling).

$$\Rightarrow VC(\text{2D linear classifiers}) = 3 = d+1 \text{ for } d\text{-dimensional linear classifiers (general result, stated)}$$

## 3. The PAC (Probably Approximately Correct) Bound - informal statement

With probability at least $1-\delta$ over the draw of an $n$-sample
training set:
$$R(h) \le \hat R(h) + \sqrt{\frac{VC(\mathcal H)\left(\log\frac{2n}{VC(\mathcal H)}+1\right)+\log\frac4\delta}{n}}$$
(exact constants vary by textbook derivation; the qualitative shape is
what matters at this course level): the generalization gap shrinks as
$O(\sqrt{VC(\mathcal H)/n})$ - **more data closes the gap, at a rate that
gets worse the more flexible your hypothesis class is**. This single
inequality is the formal backbone behind the entire practical intuition of
"complex models need more data," already built up informally via
Bias-Variance (`04_bias_variance_tradeoff.ipynb`) and empirically via
Learning Curves (`05_Model_Evaluation`).

## 4. Worked Numerical Example - how much data does a given VC dimension need?

Suppose $VC(\mathcal H)=10$, want generalization gap $\le0.05$ with 95%
confidence ($\delta=0.05$). Rearranging the bound (ignoring log factors
for a rough order-of-magnitude estimate, a standard simplification):
$$n \gtrsim \frac{VC(\mathcal H)}{\epsilon^2} = \frac{10}{0.05^2}=\frac{10}{0.0025}=4000$$
- a rough sample-complexity estimate: roughly 4000 samples needed for this
hypothesis class and this desired accuracy/confidence level. Compare
against a class with $VC=100$: needs $\approx40{,}000$ samples for the
same guarantee - a 10x increase in model flexibility costs a 10x increase
in required data, at fixed confidence, directly quantifying the
complexity/data tradeoff.

In [1]:
## verify the XOR non-shatterability claim computationally

import numpy as np
from sklearn.svm import LinearSVC

X = np.array([[0,0],[1,1],[0,1],[1,0]])
y = np.array([1,1,0,0])   # A,B vs C,D -- the un-shatterable labeling
clf = LinearSVC(max_iter=10000).fit(X, y)
print(clf.score(X, y))   # 0.5 -- chance level, confirms no linear separator exists

# Contrast: a labeling that IS achievable (3-point-style split)
y2 = np.array([1,0,0,0])   # only A is class 1 -- linearly separable
clf2 = LinearSVC(max_iter=10000).fit(X, y2)
print(clf2.score(X, y2))   # 1.0 -- perfectly separable

0.5
1.0
